# Experiment 2.2: YOLOv8m (50 epoch)

Pipeline:
1. Làm sạch dữ liệu
2. Chuyển đổi COCO sang YOLO
3. Chia tập dữ liệu
4. Huấn luyện (không cải tiến)

In [ ]:
!git clone https://github.com/Shiba-dotcom/waste-detection_project.git

In [ ]:
!pip install gdown
# Tải file từ Drive về Kaggle
!gdown --id "17Nmi2fonq31N1PZUlCZ0q7FsXf2JzGqM" -O raw.zip

# Chỉ cần tạo đến thư mục data (thư mục raw sẽ tự sinh ra khi giải nén)
!mkdir -p /kaggle/working/waste-detection_project/data

# Giải nén vào thư mục data
!unzip -q raw.zip -d /kaggle/working/waste-detection_project/data/

# Dọn dẹp file zip để tránh bị đầy bộ nhớ (Disk quota exceeded)
!rm raw.zip
!rm -rf /kaggle/working/waste-detection_project/.git

In [ ]:
%cd /kaggle/working/waste-detection_project
!pip install -r requirements.txt

In [ ]:
# Di chuyển vào đúng thư mục của script rồi mới chạy
%cd /kaggle/working/waste-detection_project/src/data_prep
!python data_cleaning.py

%cd /kaggle/working/waste-detection_project/src
!python Training_dataYolo.py

%cd /kaggle/working/waste-detection_project/src/data_prep
!python split_dataset.py

In [ ]:
import os, time, glob
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

BASE_DIR = Path.cwd().parent.parent
YAML_PATH = BASE_DIR / "data/processed/dataset.yaml"
RESULTS = BASE_DIR / "results"
RESULTS.mkdir(exist_ok=True)

print("="*55)
print("  EXPERIMENT 2.2: YOLOv8m - 50 Epoch")
print("="*55)

model_path = BASE_DIR / "models/yolov8m.pt"
model = YOLO(str(model_path))
# Train YOLOv8m, 50 epochs, no augmentations
train_results = model.train(
    data     = str(YAML_PATH),
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,
    name     = "exp2_2_yolov8m",
    project  = str(RESULTS / "runs"),
    exist_ok = True,
    verbose  = True,
)
# Evaluation
print("\n[2] Danh gia tren tap Test ...")
metrics = model.val(split="test", verbose=False)

map50 = float(metrics.box.map50)
prec = float(metrics.box.mp)
recall = float(metrics.box.mr)
map5095 = float(metrics.box.map)

print(f"\nmAP@0.5: {map50:.4f}")

val_imgs = glob.glob(str(BASE_DIR / "data/processed/images/test/**/*.*"), recursive=True)[:50]
if val_imgs:
    t0 = time.perf_counter()
    for img_path in val_imgs:
        model.predict(img_path, verbose=False)
    elapsed = (time.perf_counter() - t0) / len(val_imgs) * 1000
    print(f"Inference time (avg): {elapsed:.1f} ms/anh")
else:
    elapsed = None

result_row = {
    "Model": "YOLOv8m (exp2_2, 50 epoch)",
    "mAP@0.5 (%)": round(map50*100, 2),
    "Precision (%)": round(prec*100, 2),
    "Recall (%)": round(recall*100, 2),
    "Inference (ms)": round(elapsed, 1) if elapsed else "N/A",
}
df = pd.DataFrame([result_row])
df.to_csv(RESULTS / "ket_qua_exp2_2_yolov8m.csv", index=False)